# 1. Data, splits and graph

What the data looks like, how it is split in time, and the graph that is built from it. If this says *fixture*, the numbers describe generated data and are **not findings**.

In [ ]:
%matplotlib inline
from pipeline.notebook_support import *
import pandas as pd
import numpy as np
pd.set_option('display.width', 160)
prep = get_prepared()
cfg, df = prep.cfg, prep.df
print('FIXTURE DATA - not real results' if fixture_mode() else 'IEEE-CIS')

In [ ]:
days = (df['TransactionDT'].max() - df['TransactionDT'].min()) / 86400
pd.Series({'transactions': len(df), 'users': df['user'].nunique(), 'days': round(days, 1),
           'fraud rate': round(df['isFraud'].mean(), 4)})

## Time periods

Train, validation and test are consecutive in time, separated by an embargo so that labels that arrive late cannot leak across a boundary.

In [ ]:
from pipeline.splits import DAY, period_of
p = prep.periods
period = pd.Series(period_of(df['TransactionDT'].to_numpy(), p), name='period')
table = df.groupby(period).agg(rows=('isFraud', 'size'), fraud_rate=('isFraud', 'mean'))
table

In [ ]:
gaps = {'train->validation (days)': (p.val_start - p.t_train) / DAY, 'validation->test (days)': (p.test_start - p.val_end) / DAY}
assert all(abs(g - p.embargo / DAY) < 1e-6 for g in gaps.values()), gaps
gaps

## Account-risk population

A proxy label from fraud history up to the end of training, for users with enough history.

In [ ]:
acc = prep.account
pd.DataFrame({'users': acc.groupby('split').size(), 'eligible': acc[acc['eligible']].groupby('split').size(),
              'positives': acc[acc['eligible']].groupby('split')['label'].sum()})

## Threshold choice

The account label's fraud-rate threshold is chosen by how many positives it leaves, never by model performance. The configured value is shown against alternatives.

In [ ]:
rates = acc[acc['eligible']]['fraud_rate']
print('configured threshold:', cfg['account']['fraud_rate_threshold'])
pd.DataFrame({t: {'positives': int((rates >= t).sum()), 'prevalence': round(float((rates >= t).mean()), 4)}
              for t in (0.1, 0.2, 0.3, 0.5)}).T

## Leakage checks

Train-period fraud rows come only from train users and end by the end of the training period.

In [ ]:
tr = prep.graph['transaction'].train_mask.numpy()
assert (df.loc[tr, 'TransactionDT'] <= p.t_train).all()
assert (prep.user_split.reindex(df.loc[tr, 'user']).to_numpy() == 'train').all()
print('leakage checks passed')

## Graph

Node and edge counts of the heterogeneous graph (transaction, user, email_domain, device).

In [ ]:
g = prep.graph
display(pd.DataFrame({'nodes': {nt: g[nt].num_nodes for nt in g.node_types}}))
display(pd.DataFrame({'edges': {str(et): g[et].edge_index.shape[1] for et in g.edge_types}}))